# 🏥 Lab 3A: Linear Regression with Scikit-Learn
### Machine Learning | Effat University | Dr. Naila Marir

---

**Goal:** predict how much a person pays for **medical insurance per year**, using **linear regression**.

**Dataset:** Medical Cost dataset — 1,338 people, 6 features, 1 target (`charges`).

### 🗺️ The steps of this lab
| Step | What you do |
|---|---|
| 1 | Download the dataset and put it in Google Drive |
| 2 | Read the file in Colab |
| 3 | Explore the data (basic statistics) |
| 4 | Prepare the data |
| 5 | Learn the regression metrics |
| 6 | Train different linear regression models |
| 7 | Compare the models |
| 8 | Predict |

> 📌 After this lab, you will practise the same steps alone in **Lab 3B** (on a business dataset).

---
## Step 1 — Download the dataset and put it in Google Drive

Do these steps **by hand**, before running any code:

1. **Download** `insurance.csv` from Kaggle: https://www.kaggle.com/datasets/mirichoi0218/insurance
   *(No Kaggle account? Use this direct link, then right-click → **Save as…**:
   https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv)*
2. Open **Google Drive**: https://drive.google.com
3. In **My Drive**, create a folder `ML_Labs`, and inside it a folder `Lab3`.
4. **Upload** `insurance.csv` into `My Drive/ML_Labs/Lab3/`.

### What is in the file?
| Column | Meaning |
|---|---|
| `age` | Age of the person |
| `sex` | female / male |
| `bmi` | Body Mass Index |
| `children` | Number of children covered by the insurance |
| `smoker` | yes / no |
| `region` | US region (northeast, northwest, southeast, southwest) |
| **`charges`** | **Target:** yearly medical costs in USD |

---
## Step 2 — Read the file in Colab

### 2.1 Import the libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, SGDRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.float_format", "{:,.2f}".format)
print("✅ Libraries loaded")

### 2.2 Connect Colab to Google Drive
Run the cell, then click **Connect to Google Drive** and choose your account.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### 2.3 Read the CSV file

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/ML_Labs/Lab3/insurance.csv")
df.head()

---
## Step 3 — Explore the data (basic statistics)

### 3.1 Size of the data

In [ ]:
print("Rows   :", df.shape[0])
print("Columns:", df.shape[1])

### 3.2 Column types

In [ ]:
df.info()

> 📝 `age`, `bmi`, `children`, `charges` are **numbers**. `sex`, `smoker`, `region` are **text** → we will convert them in Step 4.

### 3.3 Missing values and duplicates

In [ ]:
print("Missing values:", df.isnull().sum().sum())
print("Duplicated rows:", df.duplicated().sum())

> 📝 No missing values, but **1 duplicated row** → we will remove it in Step 4.

### 3.4 Basic statistics

In [ ]:
df.describe()

> 📝 `mean` = average, `std` = spread, `50%` = median.
> For `charges`, the mean (~13,270) is bigger than the median (~9,380) → a few people pay **very high** charges.

### 3.5 Distribution of the charges

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df["charges"], bins=40)
plt.xlabel("Charges ($)")
plt.ylabel("Number of people")
plt.title("Distribution of the charges")
plt.show()

### 3.6 Do smokers pay more?

> ✋ **Quick check:** before running the code, guess: do smokers pay more? How many times more?

In [ ]:
df.groupby("smoker")["charges"].mean()

In [ ]:
sns.boxplot(data=df, x="smoker", y="charges")
plt.title("Charges: smokers vs. non-smokers")
plt.show()

> 📝 **Smokers pay almost 4 times more!** Smoking will be the most important feature.

### 3.7 Age, BMI and charges

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.scatterplot(data=df, x="age", y="charges", hue="smoker", ax=axes[0])
sns.scatterplot(data=df, x="bmi", y="charges", hue="smoker", ax=axes[1])
plt.show()

> 📝 **What do we see?**
> - Charges go **up with age**.
> - **Smokers with a high BMI** pay much more. Keep this in mind for Model 4!

---
## Step 4 — Prepare the data

### 4.1 Remove the duplicated row

In [ ]:
df = df.drop_duplicates()
print("Rows now:", len(df))

### 4.2 Convert text columns to numbers
`pd.get_dummies` turns each category into a **0/1 column** (e.g. `smoker_yes = 1` means the person smokes).

In [ ]:
df = pd.get_dummies(df, columns=["sex", "smoker", "region"], drop_first=True, dtype=int)
df.head()

### 4.3 Correlation with the charges
Correlation goes from **-1 to +1**. Close to **+1** → when the feature goes up, the charges go up. Close to **0** → no relation.

In [ ]:
df.corr()["charges"].sort_values(ascending=False)

> 📝 `smoker_yes` (~0.79) is the strongest, then `age` (~0.30) and `bmi` (~0.20). `sex` and `region` are almost 0.

### 4.4 Features (X) and target (y)

In [ ]:
X = df.drop(columns=["charges"])   # inputs
y = df["charges"]                  # what we want to predict
print("Features:", list(X.columns))

### 4.5 Split: 80% training, 20% test
The model **learns** on the training set and is **tested** on data it has **never seen**.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training:", len(X_train), "| Test:", len(X_test))

---
## Step 5 — Regression metrics

The **error** of one prediction = real value − predicted value.
We use **3 main metrics** to summarise all the errors:

| Metric | Formula | Meaning | Best |
|---|---|---|---|
| **MAE** | $\frac{1}{n}\sum \lvert y - \hat{y}\rvert$ | Average error, in **dollars** | lower |
| **RMSE** | $\sqrt{\frac{1}{n}\sum (y - \hat{y})^2}$ | Average error in **dollars**, but **big errors count more** | lower |
| **R²** | $1-\frac{\sum (y-\hat{y})^2}{\sum (y-\bar{y})^2}$ | % of the charges' variation explained by the model | closer to **1** |

> 💡 **R²:** 1 = perfect · 0 = no better than predicting the average · below 0 = worse than the average.
>
> ℹ️ *Good to know:* **MSE** is RMSE before the square root (unit = dollars², hard to read).
> **MAPE** gives the error in percent, and **Adjusted R²** is R² with a penalty for using many features.

### 5.1 Small example: by hand and with sklearn
5 people (charges in $1000):

In [ ]:
y_real = np.array([20, 15, 30, 25, 10])
y_pred = np.array([21, 14, 28, 26, 13])

errors = y_real - y_pred
print("Errors:", errors)

# By hand
mae  = np.mean(np.abs(errors))
rmse = np.sqrt(np.mean(errors ** 2))
r2   = 1 - np.sum(errors ** 2) / np.sum((y_real - y_real.mean()) ** 2)
print(f"By hand -> MAE = {mae:.2f} | RMSE = {rmse:.2f} | R2 = {r2:.3f}")

# With sklearn
print(f"sklearn -> MAE = {mean_absolute_error(y_real, y_pred):.2f} | "
      f"RMSE = {np.sqrt(mean_squared_error(y_real, y_pred)):.2f} | "
      f"R2 = {r2_score(y_real, y_pred):.3f}")

> 📝 RMSE (1.79) is bigger than MAE (1.60) because the biggest error (3) is **squared** → it counts more.

---
## Step 6 — Train linear regression models

> 🔗 **Explore all linear models in scikit-learn:** https://scikit-learn.org/stable/modules/linear_model.html
>
> This page lists **every type of linear regression model** in the library. Open it, look at the list,
> and click any model to see its parameters and examples. In Lab 3B you will **pick one by yourself**!

Every model in scikit-learn uses the **same 3 steps**:
```python
model = SomeModel()               # 1. create
model.fit(X_train, y_train)       # 2. train
y_pred = model.predict(X_test)    # 3. predict
```

| # | Model | Idea |
|---|---|---|
| 1 | Simple Linear Regression | **One** feature → a straight line |
| 2 | Multiple Linear Regression | **All** features |
| 3 | Linear Regression + Scaling | Same, with features on the same scale |
| 4 | Polynomial Regression | Adds squared and combined features → can fit curves |
| 5 | Ridge | Linear regression + penalty that keeps weights **small** |
| 6 | Lasso | Linear regression + penalty that can set weights to **zero** |
| 7 | ElasticNet | A mix of Ridge and Lasso |
| 8 | SGDRegressor | Linear regression trained with **gradient descent** |

### Model 1 — Simple Linear Regression (age only)
The model learns a line: $\text{charges} = w \times \text{age} + b$

In [ ]:
# Use only one column
X_train_age = X_train[["age"]]
X_test_age  = X_test[["age"]]

# 1. create
model1 = LinearRegression()

# 2. train
model1.fit(X_train_age, y_train)

# 3. predict
y_pred1 = model1.predict(X_test_age)

# 4. evaluate
print("MAE  =", round(mean_absolute_error(y_test, y_pred1)))
print("RMSE =", round(np.sqrt(mean_squared_error(y_test, y_pred1))))
print("R2   =", round(r2_score(y_test, y_pred1), 3))

print("\nSlope w     =", round(model1.coef_[0], 2))
print("Intercept b =", round(model1.intercept_, 2))

> 📝 Each extra year of age adds about **$242** per year. But **R² ≈ 0.10** → age alone explains only 10%. Let's see why:

In [ ]:
plt.scatter(X_test_age, y_test, alpha=0.5, label="Real")
plt.plot(X_test_age, y_pred1, color="red", linewidth=3, label="Model 1")
plt.xlabel("Age")
plt.ylabel("Charges ($)")
plt.legend()
plt.show()

> 📝 One straight line cannot follow the 3 groups of points. We need **more features**.

### ⚙️ A small helper function
We will repeat the same steps (create → train → predict → evaluate) for every model.
To avoid writing them again and again, we put them in a **function**. It also saves each score in a table.

In [ ]:
results = []

def evaluate(name, model, X_tr, X_te):
    model.fit(X_tr, y_train)          # 2. train
    y_pred = model.predict(X_te)      # 3. predict

    # 4. evaluate
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    r2_train = r2_score(y_train, model.predict(X_tr))

    results.append({"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2, "Train R2": r2_train})
    print(f"{name}:  MAE = {mae:,.0f} | RMSE = {rmse:,.0f} | R2 = {r2:.3f} | Train R2 = {r2_train:.3f}")
    return model

# Save Model 1 in the table
evaluate("1. Simple LR (age)", LinearRegression(), X_train_age, X_test_age);

> 💡 **Train R2** is the score on the training data. If Train R2 is **much higher** than the test R2, the model is
> **overfitting** (memorising instead of learning).

### Model 2 — Multiple Linear Regression (all features)

In [ ]:
model2 = evaluate("2. Multiple LR", LinearRegression(), X_train, X_test)

The learned weights:

In [ ]:
pd.Series(model2.coef_, index=X.columns).sort_values()

> ✋ **Quick check:** if a person becomes one year older (everything else the same), how much do the predicted charges change?
>
> <details><summary>Answer</summary>About <b>$248</b> more per year: the weight of <code>age</code>.</details>

> 📝 Being a smoker adds about **$23,000** per year! Each year of age adds ~$248, each BMI point ~$319.
> R² jumps from 0.10 to **~0.81**.

### Model 3 — Linear Regression + Scaling
`StandardScaler` puts every feature on the same scale (mean = 0, std = 1).
`make_pipeline` chains the steps: **first scale, then the model**.

In [ ]:
model3 = evaluate("3. Multiple LR + Scaling",
                  make_pipeline(StandardScaler(), LinearRegression()),
                  X_train, X_test)

> 📝 Same score as Model 2! Scaling does not change plain linear regression.
> But it is **needed** for the next models (Ridge, Lasso, ElasticNet, SGD).

### Model 4 — Polynomial Regression
`PolynomialFeatures(degree=2)` creates new features such as $\text{age}^2$ and $\text{bmi} \times \text{smoker}$.
Remember Step 3.7: smokers with high BMI pay much more → this model can learn that!

In [ ]:
model4 = evaluate("4. Polynomial (degree 2)",
                  make_pipeline(PolynomialFeatures(degree=2), StandardScaler(), LinearRegression()),
                  X_train, X_test)

> 📝 **Best score so far: R² ≈ 0.88.** The test R² is not lower than the train R² → no overfitting.

### Model 5 — Ridge
Ridge adds a penalty to keep the weights small. `alpha` = strength of the penalty (bigger → simpler model).

In [ ]:
model5 = evaluate("5. Ridge",
                  make_pipeline(StandardScaler(), Ridge(alpha=1)),
                  X_train, X_test)

### Model 6 — Lasso
Lasso can set some weights to **exactly 0** → it removes useless features by itself.

In [ ]:
model6 = evaluate("6. Lasso",
                  make_pipeline(StandardScaler(), Lasso(alpha=100)),
                  X_train, X_test)

# model6[-1] = the last step of the pipeline (the Lasso model)
weights = pd.Series(model6[-1].coef_, index=X.columns)
print("\nFeatures removed by Lasso:", list(weights[weights == 0].index))

> 📝 Lasso removed `sex_male` and `region_northwest` — features that barely affect the charges.

### Model 7 — ElasticNet
A mix of Ridge and Lasso. `l1_ratio=0.5` → 50% Lasso, 50% Ridge.

In [ ]:
model7 = evaluate("7. ElasticNet",
                  make_pipeline(StandardScaler(), ElasticNet(alpha=0.01, l1_ratio=0.5)),
                  X_train, X_test)

### Model 8 — SGDRegressor (gradient descent)
Finds the weights **step by step** with gradient descent — the same idea used to train neural networks.

In [ ]:
model8 = evaluate("8. SGD Regressor",
                  make_pipeline(StandardScaler(), SGDRegressor(random_state=42)),
                  X_train, X_test)

> 📝 Almost the same result as Model 2: two different ways to find the same line.

---
## Step 7 — Compare the models

In [ ]:
results_df = pd.DataFrame(results).set_index("Model").sort_values("RMSE")
results_df

In [ ]:
results_df["R2"].sort_values().plot(kind="barh", color="seagreen")
plt.xlabel("R² on the test set (higher is better)")
plt.show()

> ✋ **Quick check:** Model 2 and Model 3 have exactly the same score. Why?
>
> <details><summary>Answer</summary>Scaling changes the size of the weights, but not the predictions of plain linear regression.</details>

> 📝 **Conclusions**
> 1. One feature (age) is not enough → R² ≈ 0.10.
> 2. All features → R² ≈ 0.81.
> 3. **Polynomial** is the best (R² ≈ 0.88): it learned the smoker × BMI effect.
> 4. Ridge, Lasso, ElasticNet and SGD are close to plain linear regression here. They help more when there are **many** features.

---
## Step 8 — Predict

### 8.1 Real vs. predicted (best model)
Perfect predictions would be **on the red line**.

In [ ]:
best_model = model4
y_pred = best_model.predict(X_test)

plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([0, 65000], [0, 65000], color="red")
plt.xlabel("Real charges ($)")
plt.ylabel("Predicted charges ($)")
plt.show()

### 8.2 A few real and predicted values

In [ ]:
pd.DataFrame({"Real": y_test.values[:10], "Predicted": y_pred[:10].round(0)})

### 8.3 Predict for new people
Two 35-year-old women, BMI 30, 1 child, northeast region — **one non-smoker, one smoker**.

In [ ]:
new_people = pd.DataFrame({
    "age":              [35, 35],
    "bmi":              [30, 30],
    "children":         [1, 1],
    "sex_male":         [0, 0],
    "smoker_yes":       [0, 1],
    "region_northwest": [0, 0],
    "region_southeast": [0, 0],
    "region_southwest": [0, 0],
})

predictions = best_model.predict(new_people)
print(f"Non-smoker: ${predictions[0]:,.0f} per year")
print(f"Smoker    : ${predictions[1]:,.0f} per year")

---
## ✅ Summary

1. **Read & explore:** `read_csv`, `info`, `describe`, `groupby`, `corr`
2. **Prepare:** `drop_duplicates`, `get_dummies`, `train_test_split`
3. **Train & predict:** `model.fit(X_train, y_train)` → `model.predict(X_test)`
4. **Evaluate:** MAE and RMSE (error in dollars), R² (how much is explained)
5. **Models:** Linear, Polynomial, Ridge, Lasso, ElasticNet, SGD
6. **Overfitting:** compare Train R² and test R²

### 📚 Useful Links
| Topic | Link |
|---|---|
| All linear models in scikit-learn | https://scikit-learn.org/stable/modules/linear_model.html |
| Regression metrics in scikit-learn | https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics |
| pandas documentation | https://pandas.pydata.org/docs/ |
| Google Colab introduction | https://colab.research.google.com/notebooks/intro.ipynb |
| The dataset on Kaggle | https://www.kaggle.com/datasets/mirichoi0218/insurance |

📌 **Next:** open **Lab 3B** and apply all these steps by yourself on a new business dataset!